# CB-SBERT — régimen **F2** sobre **UCSD/McAuley**

Full-ranking · split random 80/10/10 · métricas `Recall/NDCG/Hit/Precision@5,10` + `Cov/Ent` + long-tail · **1 seed** (Colab gratuito).

> Carga de `ucsd_ready/*.parquet` (subir a Colab o Drive). Positivo = ownership; `playtime` min→h. 5-core(5,5).

> Plantilla = `deep_kozyriev_h3.ipynb`; modelo enchufado al harness F2 importado de `recsys_protocol.py`. Las filas MostPop/ALS son **self-check** (deben reproducir las del `*_corregido`).

In [1]:
# ====== Bootstrap: deps + clonar repo + cargar recsys_protocol ======
!pip install -q kagglehub implicit sentence-transformers
import os, sys, glob, json, math, time, csv, zipfile, shutil, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

REPO_URL = 'https://github.com/Benjaa7/Proyecto-RecSys.git'
REPO_DIR = '/content/Proyecto-RecSys'
def _locate_protocol():
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git','-C',REPO_DIR,'fetch','-q','--depth','1','origin'], check=False)
        subprocess.run(['git','-C',REPO_DIR,'reset','--hard','-q','FETCH_HEAD'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1','-q',REPO_URL,REPO_DIR], check=False)
    h3 = os.path.join(REPO_DIR, 'H3')
    if os.path.exists(os.path.join(h3, 'recsys_protocol.py')):
        return h3
    for c in ['/content','.','..','H3','../H3'] + sorted(glob.glob('/content/drive/MyDrive/*')):
        if c and os.path.exists(os.path.join(c, 'recsys_protocol.py')):
            return c
    return None
_p = _locate_protocol()
assert _p, 'No encontre recsys_protocol.py (clona el repo o sube el modulo).'
if _p not in sys.path: sys.path.insert(0, _p)

from recsys_protocol import SEED, set_global_seed, iterative_k_core
set_global_seed()
# Sección F2: usar la del módulo si está publicada; si el repo clonado trae una
# versión vieja de recsys_protocol (sin F2), caer a definiciones inline IDÉNTICAS.
try:
    from recsys_protocol import random_split_8010, paper_metrics, cat_cov_ent, recs_from_embeddings, longtail_ndcg
    print('[F2] funciones importadas de recsys_protocol')
except ImportError:
    print('[F2] recsys_protocol clonado sin sección F2 -> usando definiciones inline (idénticas al módulo)')
    def random_split_8010(inter, seed=SEED):
        inter = inter.reset_index(drop=True); rng = np.random.default_rng(seed); n = len(inter)
        inter = inter.iloc[rng.permutation(n)].reset_index(drop=True)
        n_tr, n_va = int(0.8 * n), int(0.1 * n)
        sp = np.empty(n, dtype='int8'); sp[:n_tr] = 0; sp[n_tr:n_tr + n_va] = 1; sp[n_tr + n_va:] = 2
        inter['split'] = sp; return inter
    def _dcg_f2(hits): return sum((1.0 / math.log2(i + 2)) for i, h in enumerate(hits) if h)
    def cat_cov_ent(recs, cat_map, k):
        covs, ents = [], []
        for rec in recs.values():
            cnt = {}
            for it in rec[:k]:
                for c in cat_map.get(it, ()): cnt[c] = cnt.get(c, 0) + 1
            if not cnt: covs.append(0); ents.append(0.0); continue
            covs.append(len(cnt)); tot = sum(cnt.values())
            ents.append(-sum((v / tot) * math.log2(v / tot) for v in cnt.values()))
        return (float(np.mean(covs)) if covs else 0.0, float(np.mean(ents)) if ents else 0.0)
    def paper_metrics(recs, test_items, ks=(5, 10), cat_maps=None):
        acc = {f'{m}@{k}': [] for k in ks for m in ('Recall', 'NDCG', 'Hit', 'Precision')}; n = 0
        for u, rec in recs.items():
            rel = test_items.get(u)
            if not rel: continue
            n += 1
            for k in ks:
                hits = [(1 if it in rel else 0) for it in rec[:k]]; nhit = sum(hits)
                acc[f'Recall@{k}'].append(nhit / len(rel)); acc[f'Precision@{k}'].append(nhit / k)
                acc[f'Hit@{k}'].append(1.0 if nhit > 0 else 0.0)
                idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), k)))
                acc[f'NDCG@{k}'].append(_dcg_f2(hits) / idcg if idcg > 0 else 0.0)
        out = {key: (float(np.mean(v)) if v else 0.0) for key, v in acc.items()}
        if cat_maps:
            for k in ks:
                for name, cmap in cat_maps.items():
                    cov, ent = cat_cov_ent(recs, cmap, k); out[f'Cov_{name}@{k}'] = cov; out[f'Ent_{name}@{k}'] = ent
        out['n_users'] = n; return out
    def recs_from_embeddings(e_u, e_i, umap, idx2app, users, topn, train_items_per_user, popular_list):
        out = {}; nfb = 0
        for u in users:
            seen = train_items_per_user.get(u, set()); key = str(u)
            if key not in umap:
                out[u] = [i for i in popular_list if i not in seen][:topn]; nfb += 1; continue
            scores = e_i @ e_u[umap[key]]; rec = []
            for j in np.argsort(-scores):
                a = idx2app[int(j)]
                if a not in seen:
                    rec.append(a)
                    if len(rec) >= topn: break
            out[u] = rec
        return out, nfb
    def _bucket_f2(n): return '2-5' if n <= 5 else '6-20' if n <= 20 else '21-50' if n <= 50 else '51+'
    def longtail_ndcg(recs, test_items, train_items, k=10, buckets=('2-5', '6-20', '21-50', '51+')):
        def _u_nr(rec, rel, kk):
            hits = [1 if it in rel else 0 for it in rec[:kk]]; nh = sum(hits)
            dcg = sum(1 / math.log2(i + 2) for i, h in enumerate(hits) if h)
            idcg = sum(1 / math.log2(i + 2) for i in range(min(len(rel), kk)))
            return (dcg / idcg if idcg > 0 else 0.0, nh / len(rel) if rel else 0.0)
        act = {u: _bucket_f2(len(train_items.get(u, set()))) for u in recs}; out = {}
        for b in buckets:
            us = [u for u in recs if act.get(u) == b and test_items.get(u)]
            if not us: continue
            out[b] = {'n': len(us),
                      f'NDCG@{k}': float(np.mean([_u_nr(recs[u], test_items[u], k)[0] for u in us])),
                      f'Recall@{k}': float(np.mean([_u_nr(recs[u], test_items[u], k)[1] for u in us]))}
        return out
print('bootstrap OK | numpy', np.__version__, '| pandas', pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 49.2 MB/s eta 0:00:00
[protocol] v2026-06-23b (defaults: eval sin tope max_eval_users=None + frac_train=0.30)
[protocol] seed global = 42 | numpy/random/torch (cuda=True, determinista=True)
[F2] recsys_protocol clonado sin sección F2 -> usando definiciones inline (idénticas al módulo)
bootstrap OK | numpy 2.0.2 | pandas 2.2.2


In [2]:
# ====== Config — UCSD/McAuley (5-core 5/5, split random 80/10/10) ======
DATASET = 'ucsd'
MODEL_NAME = 'CB-SBERT'
MIN_USER, MIN_GAME = 5, 5
KS = (5, 10)
ALS_FACTORS, ALS_ITERS, ALS_REG, ALS_ALPHA = 64, 15, 0.1, 40.0
N_EVAL_ANALYSIS = 200_000
TIER = 'T2'
SUBSAMPLE_USERS = None                    # UCSD es chico (~63K usuarios) -> sin submuestra
N_EXAMPLES = 3
# Cache de los parquets de UCSD. AUTOCONTENIDO: si no existen se descargan de McAuley (1 vez).
# Usa Drive si está montado (persiste entre sesiones); si no, disco local efímero.
_drive = '/content/drive/MyDrive'
UCSD_DIR = os.environ.get('UCSD_DIR') or (f'{_drive}/ucsd_ready' if os.path.isdir(_drive) else 'ucsd_ready')
os.makedirs(UCSD_DIR, exist_ok=True)
OUT = f'sbert_ucsd_h3'; os.makedirs(OUT, exist_ok=True)
print(f'CB-SBERT | UCSD | TIER={TIER} | 5-core({MIN_USER},{MIN_GAME}) | UCSD_DIR={UCSD_DIR}')


CB-SBERT | UCSD | TIER=T2 | 5-core(5,5) | UCSD_DIR=ucsd_ready


In [3]:
# ====== Carga UCSD AUTOCONTENIDA: descarga McAuley (cachea) + 5-core(5,5) + ownership + split 80/10/10 ======
# Reproducible en cualquier máquina: si faltan los parquets, se bajan de McAuley (formato dicts/linea
# -> ast.literal_eval, no JSON estricto) y se cachean en UCSD_DIR. Mismo builder que load_ucsd.ipynb.
import ast, gzip, urllib.request
_UCSD_URLS={'steam_games':'https://cseweb.ucsd.edu/~wckang/steam_games.json.gz',
            'users_items':'https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz'}
def _ucsd_stream(url):
    print('descargando', url); req=urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'}); rows=[]
    with urllib.request.urlopen(req) as resp, gzip.GzipFile(fileobj=resp) as gz:
        for line in gz: rows.append(ast.literal_eval(line.decode('utf-8')))
    return rows
def _as_list(x):
    if isinstance(x,(list,tuple)): return [str(v).strip() for v in x if v not in (None,'')]
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    s=str(x).strip(); return [s] if s else []
def _to_int(x):
    try: return int(x)
    except (TypeError,ValueError): return None
def _games_to_cat(games):
    rec=[]
    for g in games:
        aid=_to_int(g.get('id'))
        if aid is None: continue
        rec.append({'app_id':aid,'genres':_as_list(g.get('genres')),'developers':_as_list(g.get('developer')),
                    'publishers':_as_list(g.get('publisher')),
                    'name':str(g.get('app_name') or g.get('title') or f'app_{aid}'),
                    'metascore':pd.to_numeric(g.get('metascore'), errors='coerce'),  # 'NA'/int mixto -> float|NaN (evita ArrowInvalid)
                    'tags':_as_list(g.get('tags'))})
    df=pd.DataFrame(rec).drop_duplicates('app_id').reset_index(drop=True)
    df['metascore']=pd.to_numeric(df['metascore'], errors='coerce').astype('float64')   # columna homogénea para parquet
    return df
def _users_to_inter(users):
    uc,ac,pc=[],[],[]
    for u in users:
        uid=u.get('user_id')
        for it in (u.get('items') or []):
            aid=_to_int(it.get('item_id'))
            if uid is None or aid is None: continue
            pt=it.get('playtime_forever'); uc.append(str(uid)); ac.append(aid); pc.append(float(pt) if pt is not None else 0.0)
    return pd.DataFrame({'user_id':uc,'app_id':ac,'playtime':pc})
if not os.path.exists(f'{UCSD_DIR}/inter.parquet'):
    print('[ucsd] inter.parquet ausente -> construyendo desde McAuley (1 vez)...')
    _users_to_inter(_ucsd_stream(_UCSD_URLS['users_items'])).to_parquet(f'{UCSD_DIR}/inter.parquet', index=False)
if not os.path.exists(f'{UCSD_DIR}/game_categories.parquet'):
    print('[ucsd] game_categories.parquet ausente -> construyendo desde McAuley (1 vez)...')
    _games_to_cat(_ucsd_stream(_UCSD_URLS['steam_games'])).to_parquet(f'{UCSD_DIR}/game_categories.parquet', index=False)

inter_raw=pd.read_parquet(f'{UCSD_DIR}/inter.parquet')      # user_id(str), app_id(int), playtime(min)
inter_raw=inter_raw.dropna(subset=['user_id','app_id']).copy()
inter_raw['app_id']=inter_raw['app_id'].astype('int64')
inter_raw['playtime']=inter_raw['playtime'].fillna(0).clip(lower=0)/60.0   # minutos -> horas
inter_raw=inter_raw.drop_duplicates(['user_id','app_id'], keep='last', ignore_index=True)
recs_filtered=iterative_k_core(inter_raw, MIN_USER, MIN_GAME, user_col='user_id', item_col='app_id', verbose=True)
del inter_raw
pos=recs_filtered[['user_id','app_id','playtime']].copy()    # ownership = todo positivo
inter=random_split_8010(pos, SEED)
print(f'inter={len(inter):,} | usuarios={inter["user_id"].nunique():,} | juegos={inter["app_id"].nunique():,}')


[ucsd] inter.parquet ausente -> construyendo desde McAuley (1 vez)...
descargando https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz
[ucsd] game_categories.parquet ausente -> construyendo desde McAuley (1 vez)...
descargando https://cseweb.ucsd.edu/~wckang/steam_games.json.gz
  iter 1: 5,094,082 → 5,073,447
  iter 2: 5,073,447 → 5,073,447
inter=5,073,447 | usuarios=62,936 | juegos=9,192


In [4]:
# ====== Categorias/contenido UCSD (game_categories.parquet; texto=name+tags) ======
CATALOG=sorted(int(a) for a in inter['app_id'].unique()); catalog_set=set(CATALOG); n_catalog=len(CATALOG)
gc=pd.read_parquet(f'{UCSD_DIR}/game_categories.parquet'); gc['app_id']=gc['app_id'].astype('int64')
gc=gc[gc['app_id'].isin(catalog_set)].drop_duplicates('app_id')
def _lst(x):
    if isinstance(x,(list,tuple,np.ndarray)): return [str(v).strip() for v in x if str(v).strip()]
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    s=str(x).strip(); return [s] if s and s.lower()!='nan' else []
_g={int(a):_lst(v) for a,v in zip(gc['app_id'],gc['genres'])}
_d={int(a):_lst(v) for a,v in zip(gc['app_id'],gc.get('developers',pd.Series(index=gc.index,dtype=object)))}
_p={int(a):_lst(v) for a,v in zip(gc['app_id'],gc.get('publishers',pd.Series(index=gc.index,dtype=object)))}
_tg={int(a):_lst(v) for a,v in zip(gc['app_id'],gc.get('tags',pd.Series(index=gc.index,dtype=object)))}
_nm={int(a):(str(v) if isinstance(v,str) and v.strip() else f'app_{a}') for a,v in zip(gc['app_id'],gc.get('name',pd.Series(index=gc.index,dtype=object)))}
_rat=pd.to_numeric(gc.get('metascore',pd.Series(index=gc.index,dtype=object)), errors='coerce')
_ratm={int(a):(float(v) if pd.notna(v) else np.nan) for a,v in zip(gc['app_id'],_rat)}
rows=[]
for a in CATALOG:
    rows.append({'app_id':a,'name':_nm.get(a,f'app_{a}'),'metascore':_ratm.get(a,np.nan),
                 'genres':_g.get(a,[]),'developers':_d.get(a,[]),'publishers':_p.get(a,[]),'description':''})
cat=pd.DataFrame(rows)
cat['metascore']=cat['metascore'].fillna(float(pd.to_numeric(cat['metascore'],errors='coerce').median()) if cat['metascore'].notna().any() else 0.0)
desc_map={a:'' for a in CATALOG}; tags_map={a:_tg.get(a,[]) for a in CATALOG}   # UCSD sin descripcion -> name+tags
print(f'cat={len(cat):,} | con genero={sum(1 for a in CATALOG if _g.get(a))} | con tags={sum(1 for a in CATALOG if _tg.get(a))}')


cat=9,192 | con genero=7444 | con tags=7774


In [5]:
# ====== Mapas de categoria + train/test + por-usuario (comun) ======
genre_map={int(a):list(g) for a,g in zip(cat['app_id'],cat['genres'])}
dev_map  ={int(a):list(g) for a,g in zip(cat['app_id'],cat['developers'])}
pub_map  ={int(a):list(g) for a,g in zip(cat['app_id'],cat['publishers'])}
total_map={a:[('g',x) for x in genre_map.get(a,[])]+[('d',x) for x in dev_map.get(a,[])]
              +[('p',x) for x in pub_map.get(a,[])] for a in CATALOG}
CAT_MAPS={'gene':genre_map,'dev':dev_map,'pub':pub_map,'total':total_map}
name_map={int(a):n for a,n in zip(cat['app_id'],cat['name'])}

train=inter[inter['split']==0]; test=inter[inter['split']==2]
train_items_per_user=train.groupby('user_id')['app_id'].apply(lambda s:set(int(x) for x in s)).to_dict()
test_items_per_user ={u:set(int(x) for x in g) for u,g in test.groupby('user_id')['app_id']}
eval_users=[u for u in test_items_per_user if u in train_items_per_user]
print(f'train={len(train):,} test={len(test):,} | eval usuarios={len(eval_users):,} | '
      f'positivos/usuario(medio)={np.mean([len(test_items_per_user[u]) for u in eval_users]):.2f}')


train=4,058,757 test=507,346 | eval usuarios=57,424 | positivos/usuario(medio)=8.84


In [6]:
# ====== MostPop + ALS (self-check) + eval_set fijo ======
import scipy.sparse as _sp
from implicit.als import AlternatingLeastSquares
pop=train.groupby('app_id').size().to_dict()
popular_list=[it for it,_ in sorted(pop.items(), key=lambda kv:(-kv[1],kv[0])) if it in catalog_set]
TOPN=max(KS)

als_users=sorted(train['user_id'].unique().tolist())
_u2i={u:i for i,u in enumerate(als_users)}; _a2i={a:i for i,a in enumerate(CATALOG)}
idx2app_als={i:a for a,i in _a2i.items()}
_tr=train[train['app_id'].isin(_a2i)]
_rows=_tr['user_id'].map(_u2i).to_numpy(); _cols=_tr['app_id'].map(_a2i).to_numpy()
_conf=(1.0+ALS_ALPHA*np.log1p(_tr['playtime'].fillna(0).clip(lower=0).to_numpy())).astype('float32')
_ui=_sp.csr_matrix((_conf,(_rows,_cols)), shape=(len(als_users),len(CATALOG)))
als=AlternatingLeastSquares(factors=ALS_FACTORS, regularization=ALS_REG, iterations=ALS_ITERS, random_state=SEED, use_gpu=False)
als.fit(_ui)
als_uf=np.asarray(als.user_factors); als_if=np.asarray(als.item_factors)
als_umap={str(u):_u2i[u] for u in als_users}
print('ALS listo:', als_uf.shape, als_if.shape)

# eval_set fijo (mismo criterio que el *_corregido: seed 42, tope N_EVAL_ANALYSIS)
if TIER=='T1': eval_users=eval_users[:2000]
if N_EVAL_ANALYSIS and len(eval_users)>N_EVAL_ANALYSIS:
    _rng=np.random.default_rng(SEED)
    eval_set=sorted(_rng.choice(np.array(eval_users), size=N_EVAL_ANALYSIS, replace=False).tolist())
else:
    eval_set=list(eval_users)
print(f'eval_set: {len(eval_set):,} usuarios')

recs_fixed={}
recs_fixed['Most Popular']={u:[i for i in popular_list if i not in train_items_per_user.get(u,set())][:TOPN] for u in eval_set}
recs_fixed['ALS'],_=recs_from_embeddings(als_uf,als_if,als_umap,idx2app_als,eval_set,TOPN,train_items_per_user,popular_list)
metrics_fixed={m:paper_metrics(recs_fixed[m],test_items_per_user,ks=KS,cat_maps=CAT_MAPS) for m in recs_fixed}
for m in recs_fixed:
    d=metrics_fixed[m]
    print(f'  {m:13s} R@5={d["Recall@5"]:.4f} NDCG@5={d["NDCG@5"]:.4f} NDCG@10={d["NDCG@10"]:.4f} '
          f'Hit@10={d["Hit@10"]:.4f} (self-check vs *_corregido)')


  0%|          | 0/15 [00:00<?, ?it/s]

ALS listo: (62936, 64) (9192, 64)
eval_set: 57,424 usuarios
  Most Popular  R@5=0.1191 NDCG@5=0.1672 NDCG@10=0.1655 Hit@10=0.5752 (self-check vs *_corregido)
  ALS           R@5=0.1050 NDCG@5=0.1246 NDCG@10=0.1443 Hit@10=0.5606 (self-check vs *_corregido)


## Modelo: CB-SBERT

In [7]:
# ====== CB-SBERT — embeddings de contenido; perfil = media ponderada por confianza ======
from sentence_transformers import SentenceTransformer
import torch
_dev='cuda' if torch.cuda.is_available() else 'cpu'
def _build_text(a):
    parts=[name_map.get(a,'')]
    d=desc_map.get(a,'')
    if d: parts.append(d)
    tg=tags_map.get(a,[])
    if tg: parts.append('Tags: '+', '.join(tg))
    txt='. '.join(p for p in parts if p).strip()
    return txt or (name_map.get(a,'') or 'unknown game')
_cache=f'{OUT}/sbert_item_emb.npy'
if os.path.exists(_cache):
    item_emb=np.load(_cache); print('embeddings SBERT leidos de cache:', item_emb.shape)
else:
    sbert=SentenceTransformer('all-MiniLM-L6-v2', device=_dev)
    item_text=[_build_text(a) for a in CATALOG]
    item_emb=sbert.encode(item_text, batch_size=256, show_progress_bar=True,
                          convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    np.save(_cache, item_emb); print('embeddings SBERT:', item_emb.shape, '-> cache', _cache)

# perfil de usuario = media de embeddings de su historial de train, ponderada por log1p(playtime)
user_emb=np.zeros((len(als_users), item_emb.shape[1]), dtype='float32')
for u,g in _tr.groupby('user_id'):
    idxs=[_a2i[a] for a in g['app_id']]            # _tr ya filtrado a app_id en _a2i -> alineado
    if not idxs: continue
    w=np.log1p(g['playtime'].fillna(0).clip(lower=0).to_numpy()).astype('float32')
    w=np.where(w<=0,1e-6,w); w=w/w.sum()
    prof=w@item_emb[idxs]; nrm=np.linalg.norm(prof)
    if nrm>0: prof=prof/nrm
    user_emb[_u2i[u]]=prof
recs_fixed['CB-SBERT'],_nfb=recs_from_embeddings(user_emb,item_emb,als_umap,idx2app_als,eval_set,TOPN,train_items_per_user,popular_list)
metrics_fixed['CB-SBERT']=paper_metrics(recs_fixed['CB-SBERT'],test_items_per_user,ks=KS,cat_maps=CAT_MAPS)
d=metrics_fixed['CB-SBERT']
print(f'CB-SBERT  R@5={d["Recall@5"]:.4f} NDCG@5={d["NDCG@5"]:.4f} NDCG@10={d["NDCG@10"]:.4f} '
      f'Hit@10={d["Hit@10"]:.4f} | cold-fallback={_nfb}')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

embeddings SBERT: (9192, 384) -> cache sbert_ucsd_h3/sbert_item_emb.npy
CB-SBERT  R@5=0.0128 NDCG@5=0.0152 NDCG@10=0.0168 Hit@10=0.0846 | cold-fallback=0


## Análisis (A accuracy · B diversidad · C long-tail · D ejemplos)

In [8]:
# ====== A/B/C/D (accuracy, diversidad, long-tail, ejemplos) + guardar + imprimir ======
import json as _json
MODELS=['Most Popular','ALS',MODEL_NAME]
_MET=['Recall@5','NDCG@5','Hit@5','Precision@5','Recall@10','NDCG@10']
dfA=pd.DataFrame([[m]+[f'{metrics_fixed[m][x]:.4f}' for x in _MET] for m in MODELS], columns=['Modelo']+_MET)
print('=== A. Accuracy (split 80/10/10, full-ranking, 1 seed) ===')
print(dfA.to_string(index=False))

DIV=[f'Cov_total@{k}' for k in KS]+[f'Cov_gene@{k}' for k in KS]+[f'Ent_gene@{k}' for k in KS]
dfB=pd.DataFrame({m:{c:f'{metrics_fixed[m][c]:.4f}' for c in DIV} for m in MODELS}).T[DIV]
print('\n=== B. Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia) ===')
print(dfB.to_string())

lt={m:longtail_ndcg(recs_fixed[m],test_items_per_user,train_items_per_user,k=10) for m in MODELS}
buckets=['2-5','6-20','21-50','51+']
rowsC=[]
for b in buckets:
    row={'actividad':b}
    for m in MODELS: row[m]=(f'{lt[m][b]["NDCG@10"]:.4f}' if b in lt[m] else '')
    rowsC.append(row)
dfC=pd.DataFrame(rowsC)[['actividad']+MODELS]
print('\n=== C. Long-tail: NDCG@10 por actividad del usuario ===')
print(dfC.to_string(index=False))
print('n usuarios/bucket:', {b:(lt['Most Popular'][b]['n'] if b in lt['Most Popular'] else 0) for b in buckets})

def _u_nr5(rec,rel):
    hits=[1 if it in rel else 0 for it in rec[:5]]; nh=sum(hits)
    dcg=sum(1/math.log2(i+2) for i,h in enumerate(hits) if h)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),5)))
    return (dcg/idcg if idcg>0 else 0.0, nh/len(rel) if rel else 0.0)
_act={u:('2-5' if len(train_items_per_user.get(u,set()))<=5 else '6-20' if len(train_items_per_user.get(u,set()))<=20
         else '21-50' if len(train_items_per_user.get(u,set()))<=50 else '51+') for u in eval_set}
ej=[f'(ejemplos seed {SEED})']; _chosen=[]
for b in buckets:
    for u in [x for x in eval_set if _act[x]==b and test_items_per_user.get(x)]:
        if all(u in recs_fixed[m] for m in MODELS) and any(any(it in test_items_per_user[u] for it in recs_fixed[m][u][:5]) for m in MODELS):
            _chosen.append((b,u)); break
    if len(_chosen)>=N_EXAMPLES: break
def _nm(items): return [name_map.get(a,str(a)) for a in items]
for b,u in _chosen[:N_EXAMPLES]:
    rel=test_items_per_user[u]; hist=sorted(train_items_per_user.get(u,set()))
    ej.append(f'\n### Usuario {u} (actividad {b}, {len(hist)} juegos en historial)')
    ej.append('- Perfil (muestra): '+', '.join(_nm(hist[:6])))
    ej.append('- Test (a acertar): '+', '.join(_nm(sorted(rel))))
    for m in MODELS:
        nd,rc=_u_nr5(recs_fixed[m][u],rel)
        marks=[name_map.get(a,str(a))+(' ✓' if a in rel else '') for a in recs_fixed[m][u][:5]]
        ej.append(f'  - **{m}** (R@5={rc:.2f}, NDCG@5={nd:.2f}): '+', '.join(marks))
ej_txt='\n'.join(ej)
print('\n=== D. Ejemplos ===\n'+ej_txt)

dfA.to_csv(f'{OUT}/accuracy.csv', index=False); dfB.to_csv(f'{OUT}/diversidad.csv'); dfC.to_csv(f'{OUT}/longtail.csv', index=False)
open(f'{OUT}/ejemplos.md','w',encoding='utf-8').write(ej_txt)
_full={'dataset':DATASET,'model':MODEL_NAME,'seed':SEED,'tier':TIER,'n_eval':len(eval_set),
       'metrics':{m:{k:(float(v) if isinstance(v,(int,float,np.floating)) else v) for k,v in metrics_fixed[m].items()} for m in MODELS}}
_json.dump(_full, open(f'{OUT}/metricas_full.json','w'), indent=2)
print('\n'+'#'*72+'\n# RESPALDO EN TEXTO (todo impreso por si no se descargan archivos)\n'+'#'*72)
print('\n== accuracy.csv ==\n'+dfA.to_csv(index=False))
print('== diversidad.csv ==\n'+dfB.to_csv())
print('== longtail.csv ==\n'+dfC.to_csv(index=False))
print('== metricas_full.json ==\n'+_json.dumps(_full, indent=2))
try:
    shutil.make_archive(OUT,'zip',OUT)
    from google.colab import files; files.download(OUT+'.zip'); print('(zip de descarga generado)')
except Exception as e:
    print('(descarga automatica no disponible:', repr(e), '-> todo esta impreso arriba)')


=== A. Accuracy (split 80/10/10, full-ranking, 1 seed) ===
      Modelo Recall@5 NDCG@5  Hit@5 Precision@5 Recall@10 NDCG@10
Most Popular   0.1191 0.1672 0.4663      0.1253    0.1622  0.1655
         ALS   0.1050 0.1246 0.3823      0.0952    0.1684  0.1443
    CB-SBERT   0.0128 0.0152 0.0527      0.0110    0.0190  0.0168

=== B. Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia) ===
             Cov_total@5 Cov_total@10 Cov_gene@5 Cov_gene@10 Ent_gene@5 Ent_gene@10
Most Popular     11.1765      17.4323     5.4149      7.0177     2.1906      2.4529
ALS              13.3065      22.1631     5.6837      7.4945     2.2216      2.5337
CB-SBERT         14.9704      25.6404     5.8534      7.6613     2.2485      2.5047

=== C. Long-tail: NDCG@10 por actividad del usuario ===
actividad Most Popular    ALS CB-SBERT
      2-5       0.2235 0.1999   0.0359
     6-20       0.1768 0.1900   0.0259
    21-50       0.1561 0.1541   0.0148
      51+       0.1628 0.1110   0.0125
n usuarios/b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(zip de descarga generado)
